# FROG: Fine-Tuned LegalGraphRAG Implementation

## Setup and Dependencies

In [ ]:
# Install required packages
!pip install -q unsloth==2025.5.7
!pip install -q sentence-transformers SPARQLWrapper weaviate-client googletrans-py==4.0.0 rdflib \
           langchain langchain-core langchain-community langchain-huggingface==0.1.2 pydantic nltk dotenv
!pip install -q protobuf==3.20.*

In [ ]:
# Create directory structure
import os

os.makedirs('data/legal_ontology', exist_ok=True)
os.makedirs('logs', exist_ok=True)

In [ ]:
# Download Legal ontology data and test datasets
!wget -O data/legal_ontology/final_result.ttl https://raw.githubusercontent.com/gansixeneh/FROG-2.0/refs/heads/dataset/legal/modified_data-lex2kg.ttl
!wget -O train_template.json -nv https://raw.githubusercontent.com/gansixeneh/FROG-2.0/refs/heads/dataset/legal/legal_train.json
!wget -O test_template.json -nv https://raw.githubusercontent.com/gansixeneh/FROG-2.0/refs/heads/dataset/legal/legal_test.json
!wget -O train_rw.json -nv https://raw.githubusercontent.com/gansixeneh/FROG-2.0/refs/heads/dataset/legal/rw/legal_train.json
!wget -O test_rw.json -nv https://raw.githubusercontent.com/gansixeneh/FROG-2.0/refs/heads/dataset/legal/rw/legal_test.json

In [ ]:
# Set up deterministic behavior for reproducibility
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

import torch
import random
import numpy as np

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

In [ ]:
# Unsloth imports for optimized model loading
from unsloth import FastLanguageModel
from unsloth import is_bfloat16_supported

# For fine-tuning
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset
from sklearn.model_selection import train_test_split

import torch
import pandas as pd
import os
import json
import gc
import re
import nltk
import numpy as np
from dotenv import load_dotenv
from copy import deepcopy
from xml.sax.saxutils import escape
from IPython.display import HTML, display
from SPARQLWrapper import SPARQLWrapper, JSON
import requests
from sentence_transformers import SentenceTransformer
from googletrans import Translator
import googletrans
from typing import List, Optional, Dict, Any, Tuple, Union, Literal
from pydantic import BaseModel, Field
from transformers import pipeline
from tqdm import tqdm
import matplotlib.pyplot as plt
from rdflib import Graph

# langchain imports
from langchain_core.output_parsers import StrOutputParser
from langchain.chains import LLMChain
from langchain.output_parsers import (
    ResponseSchema,
    StructuredOutputParser,
    PydanticOutputParser,
)
from langchain_huggingface.llms import HuggingFacePipeline
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate,
    MessagesPlaceholder,
)
from langchain.output_parsers.prompts import NAIVE_FIX_PROMPT

# Download NLTK data
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer
from nltk import ngrams

In [ ]:
def clear_memory():
    """Aggressively clear GPU memory"""
    import gc
    import torch
    
    # Clear CUDA cache
    torch.cuda.empty_cache()
    
    # Force garbage collection
    gc.collect()
    
    # Add synchronization
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        
    # Log memory status
    if torch.cuda.is_available():
        print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
        print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

## Define Helper Functions


In [ ]:
# Helper functions
def replace_using_dict(original_string, replacements) -> str:
    for old, new in replacements.items():
        original_string = original_string.replace(old, new)
    return original_string

def separate_camel_case(s) -> str:
    separated = re.sub("([a-z])([A-Z])", r"\1 \2", s)
    return separated

def fix_query_spacing(query: str) -> str:
    # Add a space after 'select' if it's followed immediately by a variable (e.g., ?x, ?y)
    query = re.sub(r"(?i)\b(select)(\?\w+)", r"\1 \2", query)
    # Add a space before any variable in a predicate-object pair
    query = re.sub(r"(\w+:\w+)(\?\w+)", r"\1 \2", query)
    # Remove '^^xsd:string'
    query = re.sub(r"\^\^xsd:string", "", query, flags=re.IGNORECASE)
    return query

def legal_entity_label(url):
    parts = url.strip('/').split('/')
    transformed_parts = []
    
    month_mapping = {
        "January": "Januari", "February": "Februari", "March": "Maret", "April": "April", "May": "Mei", "June": "Juni",
        "July": "Juli", "August": "Agustus", "September": "September", "October": "Oktober", "November": "November", "December": "Desember"
    }
    
    for i, part in enumerate(parts):
        if part == "lex2kg":
            transformed_parts = []
            continue
        if part == "uu":
            transformed_parts.append("UU")
        elif part.isdigit() and len(part) <= 2:
            transformed_parts.append(f"no {part}")
        elif part.isdigit() and len(part) == 4 and int(part) >= 1945:
            transformed_parts.append(f"tahun {part}")
        elif part.isdigit() and len(part) == 8:
            try:
                date_obj = datetime.strptime(part, "%Y%m%d")
                formatted_date = date_obj.strftime("%-d %B %Y")
                for eng, indo in month_mapping.items():
                    formatted_date = formatted_date.replace(eng, indo)
                transformed_parts.append(formatted_date)
            except ValueError:
                transformed_parts.append(part)
        elif part.isdigit():
            num = str(int(part))
            transformed_parts.append(num)
        else:
            transformed_parts.append(separate_camel_case(part).lower())
    
    return ' '.join(transformed_parts)

def legal_property_label(x):
    if "http" in x:
        x = x.split("/")[-1]
    else:
        x = x.split(":")[-1]
    return separate_camel_case(x).lower()

## Legal Verbalization Implementation

In [ ]:
# Legal Verbalization class
class LegalVerbalization:
    SENTENCE_TEMPLATE = "{p} dari {s} adalah {o}"
    SENTENCE_TEMPLATE_BAGIAN_DARI = "{s} merupakan bagian dari {o}"
    SENTENCE_TEMPLATE_ME = "Yang {p} {o} adalah {s}"
    SENTENCE_TEMPLATE_DI = "{s} {p} {o}"
    MANUAL_MAPPING_DICT = {"_": " "}
    PO_TEMPLATE = """
SELECT DISTINCT
    ?p ?o
WHERE {{
    <{entity}> ?p ?o. 
    FILTER(STRSTARTS(STR(?p), STR(lex2kg-o:)))
}}
"""
    SP_TEMPLATE = """
SELECT DISTINCT
    ?s ?p
WHERE {{ 
        ?s ?p <{entity}>.
        FILTER(STRSTARTS(STR(?p), STR(lex2kg-o:)))
}}
"""

    def __init__(
        self,
        model_name="jinaai/jina-embeddings-v3",
        model_kwargs={"trust_remote_code": True},
        query_model_encode_kwargs={},
        passage_model_encode_kwargs={},
        turtle_file_path=None,
    ) -> None:
        self.model_name = model_name
        self.query_model_encode_kwargs = query_model_encode_kwargs
        self.passage_model_encode_kwargs = passage_model_encode_kwargs
        self.model = SentenceTransformer(model_name, **model_kwargs)
        self.model.eval()
        
        # Load knowledge graph
        if turtle_file_path:
            self.graph = Graph().parse(turtle_file_path)
        else:
            self.graph = None

    def get_po(self, entity: str) -> pd.DataFrame:
        query = self.PO_TEMPLATE.format(entity=entity)
        response = self.graph.query(query)
        df = pd.DataFrame(response.bindings)
        if not df.empty:
            df.columns = [str(col) for col in df.columns]
            
            df["sLabel"] = legal_entity_label(entity)
            df["pLabel"] = df["p"].apply(legal_property_label)
            df["oLabel"] = df["o"].apply(legal_entity_label)
            
            cols = ["p", "o", "sLabel", "pLabel", "oLabel"]
            df = df[cols]
            
            for col in cols:
                df[col] = df[col].apply(lambda x: str(x))
        if df.empty:
            return pd.DataFrame(columns=["p", "o", "sLabel", "pLabel", "oLabel"])
        return df

    def get_sp(self, entity: str) -> pd.DataFrame:
        query = self.SP_TEMPLATE.format(entity=entity)
        response = self.graph.query(query)
        df = pd.DataFrame(response.bindings)
        if not df.empty:
            df.columns = [str(col) for col in df.columns]
            
            df["sLabel"] = df["s"].apply(legal_entity_label)
            df["pLabel"] = df["p"].apply(legal_property_label)
            df["oLabel"] = legal_entity_label(entity)
            
            cols = ["s", "p", "sLabel", "pLabel", "oLabel"]
            df = df[cols]
            
            for col in cols:
                df[col] = df[col].apply(lambda x: str(x))
        if df.empty:
            return pd.DataFrame(columns=["s", "p", "sLabel", "pLabel", "oLabel"])
        return df
        
    def get_kalimat(self, label_s, label_p, label_o):
        result = self.SENTENCE_TEMPLATE.format(
            s=str(label_s), p=str(label_p), o=str(label_o)
        )
        
        if label_p == "bagian dari":
            result = self.SENTENCE_TEMPLATE_BAGIAN_DARI.format(
                s=str(label_s), o=str(label_o)
            )
        
        if label_p[:2] == "me":
            result = self.SENTENCE_TEMPLATE_ME.format(
                s=str(label_s), p=str(label_p), o=str(label_o)
            )
        
        if label_p[:2] == "di":
            result = self.SENTENCE_TEMPLATE_DI.format(
                s=str(label_s), p=str(label_p), o=str(label_o)
            )
        
        return result

    def get_list_of_candidates(self, entity: str):
        po, sp = self.get_po(entity), self.get_sp(entity)
        candidates = dict()

        # Process predicate-object pairs
        curr_p = None
        for _, (p, o, sLabel, pLabel, oLabel) in po.iterrows():
            label_s = sLabel if sLabel else replace_using_dict(entity.split("/")[-1], self.MANUAL_MAPPING_DICT)
            label_p = pLabel if pLabel else separate_camel_case(p.split("/")[-1])

            if label_p != curr_p:
                curr_p = label_p
                if o.startswith("http"):
                    label_o = oLabel if oLabel else replace_using_dict(o.split("/")[-1], self.MANUAL_MAPPING_DICT)
                else:
                    label_o = o
                candidates[(p, "po")] = self.get_kalimat(label_s, label_p, label_o)

        # Process subject-predicate pairs
        curr_p = None
        for _, (s, p, sLabel, pLabel, oLabel) in sp.iterrows():
            label_s = sLabel if sLabel else replace_using_dict(s.split("/")[-1], self.MANUAL_MAPPING_DICT)
            label_p = pLabel if pLabel else separate_camel_case(p.split("/")[-1])
            label_o = oLabel if oLabel else replace_using_dict(entity.split("/")[-1], self.MANUAL_MAPPING_DICT)

            if label_p != curr_p:
                curr_p = label_p
                candidates[(p, "sp")] = self.get_kalimat(label_s, label_p, label_o)

        return candidates, po, sp

    def run(self, question: str, entity: str, output_uri=False) -> tuple[list[dict[str, str]], float]:
        # Get candidate sentences
        list_of_candidates, po, sp = self.get_list_of_candidates(entity)
        cands = list(list_of_candidates.values())
        if not cands:  # Handle empty candidates
            return [], 0.0
            
        # Encode question and candidates
        question_embed = self.model.encode(
            question,
            **self.query_model_encode_kwargs
        )
        passages_embed = self.model.encode(
            cands,
            **self.passage_model_encode_kwargs
        )

        # Find most similar candidate
        similarities = self.model.similarity(question_embed, passages_embed).numpy().flatten()
        similar_index = np.argmax(similarities)
        similar_score = max(similarities)

        # Extract results based on the most similar property
        property_used, type = list(list_of_candidates.keys())[similar_index]
        result = []
        
        # Process predicate-object pairs
        if type == "po":
            for _, (p, o, _, pLabel, oLabel) in po[po["p"] == property_used].iterrows():
                label_p = pLabel if pLabel else separate_camel_case(p.split("/")[-1])
                if o.startswith("http"):
                    label_o = oLabel if oLabel else replace_using_dict(o.split("/")[-1], self.MANUAL_MAPPING_DICT)
                else:
                    label_o = o
                result.append({"val": o if output_uri else label_o})
        
        # Process subject-predicate pairs
        if type == "sp":
            for _, (s, p, sLabel, pLabel, _) in sp[sp["p"] == property_used].iterrows():
                label_s = sLabel if sLabel else replace_using_dict(s.split("/")[-1], self.MANUAL_MAPPING_DICT)
                result.append({"val": s if output_uri else label_s})
            
        return result, similar_score

## Model Loading and Fine-Tuning Utilities

In [ ]:
from huggingface_hub import HfApi, create_repo
import shutil

model_name = "Qwen/Qwen2.5-Coder-7B-Instruct"

max_seq_length = 4096
save_to_gguf = False
hf_token = "MY_HF_TOKEN"

# Model loading and memory management utility
def load_model(model_name: str, load_in_4bit=True):
    """
    Load a model with memory optimization
    """
    # Load model with unsloth for reduced memory usage
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=max_seq_length,
        dtype=None,  # Auto detection: Float16 for Tesla T4/V100, Bfloat16 for Ampere+
        load_in_4bit=load_in_4bit,  # 4-bit quantization
    )
    
    # Set template for some models
    if not hasattr(tokenizer, "chat_template") and tokenizer.chat_template is None and 'qwen' in model_name.lower():
        tokenizer.chat_template = "<|im_start|>system\n{{ messages[0]['content'] }}<|im_end|>\n{% for message in messages[1:] %}<|im_start|>{{ message['role'] }}\n{{ message['content'] }}<|im_end|>\n{% endfor %}"
        print("Qwen chat template has been set.")
        
    return model, tokenizer

def get_pipeline(model, tokenizer, max_new_tokens=512):
    """
    Create a pipeline from a model and tokenizer
    """
    pipe = pipeline(
        "text-generation",
        model=model, 
        tokenizer=tokenizer,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        top_k=None,
        top_p=None,
        temperature=None,
        torch_dtype=torch.bfloat16 if is_bfloat16_supported() else torch.float16,
        return_full_text=False
    )
    
    return HuggingFacePipeline(pipeline=pipe)

def finetune_entity_property(model_name, train_dataset, val_dataset, output_dir, num_epochs, dataset_type: Literal["template", "rw"]):
    """
    Fine-tune a model with LoRA
    """
    model, tokenizer = load_model(model_name)
    
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
        use_rslora=False,
        loftq_config=None,
    )
    
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        dataset_num_proc=2,
        packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=4,
            warmup_steps=5,
            # num_train_epochs=num_epochs,
            max_steps = 2,
            learning_rate=2e-5,
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=42,
            output_dir="outputs",
            report_to="none",
            
            eval_strategy="steps",
            save_strategy="steps",
            save_steps=20,
            eval_steps=20,
            logging_steps=10,
            
            # Checkpoint management
            save_total_limit=5,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
        ),
    )

    trainer.train()

    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    if save_to_gguf:
        save_repo_name = f"gansixeneh/{model_name.split('/')[-1]}"
        repo_name = f"{save_repo_name}-legal-sparql-{dataset_type}"
        create_repo(repo_name, token=hf_token, exist_ok=True)
        model.push_to_hub_gguf(repo_name, tokenizer, quantization_method = "q4_k_m", token=hf_token)
        shutil.rmtree('/kaggle/working/gansixeneh')
    
    return model, tokenizer

def finetune_sparql(model_name, train_dataset, val_dataset, output_dir, num_epochs, dataset_type: Literal["template", "rw"]):
    """
    Input adapter_path if you want to save your adapter weights
    """    
    model, tokenizer = load_model(model_name)

    model = FastLanguageModel.get_peft_model(
        model,
        r = 64,  # Increased from 32 for more capacity to handle structured syntax
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj",],
        lora_alpha = 64,  # Matching r
        lora_dropout = 0.1,  # Increased for better generalization 
        bias = "all",  # Changed from "none" to better learn syntax patterns
        use_gradient_checkpointing = "unsloth",
        random_state = 42,
        use_rslora = True,  # Keep RSLoRA for stability
        loftq_config = None,
    )
    
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = train_dataset,
        eval_dataset = val_dataset,
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        dataset_num_proc = 4,  # Increased for faster processing
        packing = True,  # Enable packing for efficiency with varied sequence lengths
        args = TrainingArguments(
            # Batch sizing
            per_device_train_batch_size = 1,  # Reduced to focus on quality over speed
            per_device_eval_batch_size = 1,
            gradient_accumulation_steps = 16,  # Increased for larger effective batch
            
            # Learning rate
            learning_rate = 1e-5,  # Lower learning rate for careful convergence
            weight_decay = 0.1,  # Increased regularization to combat overfitting
            
            # Schedule
            # num_train_epochs = num_epochs,
            max_steps = 2,
            lr_scheduler_type = "polynomial",  # Better than cosine for syntax learning
            warmup_ratio = 0.1,  # Longer warmup (10% of training) 
            
            # Precision
            fp16 = not is_bfloat16_supported(),
            bf16 = is_bfloat16_supported(),
            optim = "adamw_8bit",
            
            # Evaluation
            eval_strategy = "steps",
            save_strategy = "steps",
            eval_steps = 25,  # More frequent evaluation
            save_steps = 25,
            logging_steps = 5,
            
            # Checkpoint management
            save_total_limit = 5,  # Keep more checkpoints
            load_best_model_at_end = True,
            metric_for_best_model = "eval_loss",
            greater_is_better = False,
            
            # Additional settings
            group_by_length = True,
            gradient_checkpointing = True,
            dataloader_drop_last = False,
            
            # Add label smoothing for SPARQL syntax tokens
            label_smoothing_factor = 0.05,
            
            # Output
            output_dir = "outputs",
            report_to = "tensorboard",
            seed = 42,
        ),
    )
    
    trainer.train()

    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    if save_to_gguf:
        save_repo_name = f"gansixeneh/{model_name.split('/')[-1]}"
        repo_name = f"{save_repo_name}-legal-sparql-{dataset_type}"
        create_repo(repo_name, token=hf_token, exist_ok=True)
        model.push_to_hub_gguf(repo_name, tokenizer, quantization_method = "q4_k_m", token=hf_token)
        shutil.rmtree('/kaggle/working/gansixeneh')
    
    return model, tokenizer

## Prepare Datasets

In [ ]:
# Load and process datasets
def load_dataset_sparql(path_to_data: str):
    with open(path_to_data, "r", encoding="utf-8") as file:
        raw = json.load(file)
        
    data = []
    for item in raw:
        data.append({
            "question": item['question'],
            "entities_matches": item['entities_matches'],
            "properties_matches": item['properties_matches'],
            "sparql": item['sparql'],
            "thoughts": item.get('thoughts', None)
        })
    
    df = pd.DataFrame(data)
    print(f"Extracted {len(df)} data from {path_to_data}")

    return df

def load_dataset_entity_prop(path_to_data: str):
    with open(path_to_data, "r", encoding="utf-8") as file:
        raw = json.load(file)
        
    data = []
    for item in raw:
        data.append({
            "question": item['question'],
            "entities": item['entities'],
            "properties": item['properties'],
        })
    
    df = pd.DataFrame(data)
    print(f"Extracted {len(df)} data from {path_to_data}")

    return df

# Update load_test_data function to include complexity
def load_test_data(path_to_data):
    with open(path_to_data, "r", encoding="utf-8") as file:
        raw = json.load(file)
    
    data = []
    for item in raw:
        data.append({
            "question": item['question'],
            "sparql": item.get('sparql', ''),
            "complexity": item.get('complexity', 'unknown')  # Add complexity field
        })
    
    return pd.DataFrame(data)
    
# Load all datasets
test_df = load_test_data("test_template.json")
entity_prop_train_df = load_dataset_entity_prop("train_template.json")
sparql_train_df = load_dataset_sparql("train_template.json")

# Display information
print(f"Loaded {len(test_df)} test questions")
print(f"Loaded {len(entity_prop_train_df)} entity-property training examples")
print(f"Loaded {len(sparql_train_df)} SPARQL training examples")

# Split train/validation for fine-tuning
entity_prop_train_df, entity_prop_val_df = train_test_split(entity_prop_train_df, test_size=0.2, random_state=42)
sparql_train_df, sparql_val_df = train_test_split(sparql_train_df, test_size=0.2, random_state=42)

print(f"Entity-property train/val split: {len(entity_prop_train_df)}/{len(entity_prop_val_df)}")
print(f"SPARQL train/val split: {len(sparql_train_df)}/{len(sparql_val_df)}")

## Prepare Training Data for Fine-Tuning


In [ ]:
def check_template(model_name: str):
    """
    Load a model to get tokenizer and check if it has a chat template
    Returns both the boolean flag and the tokenizer for reuse
    """
    print(f"Loading tokenizer and checking chat template for {model_name}...")
    
    # Load model and tokenizer
    model, tokenizer = load_model(model_name)
    
    # Check if the tokenizer has a chat template
    has_template = hasattr(tokenizer, "chat_template") and tokenizer.chat_template is not None
    print(f"Model has chat template: {has_template}")
    
    # Clean up model but keep tokenizer
    del model
    clear_memory()
    clear_memory()
    
    return has_template, tokenizer

# Set the global variable before dataset preparation
has_chat_template, tokenizer = check_template(model_name)
print(f"Global has_chat_template set to: {has_chat_template}")

In [ ]:
# Format entity and property extraction data for fine-tuning
def format_entity_prop_instruction(row):
    system_prompt = """You are an expert entity and property extractor for university course knowledge graph querying. Your task is to analyze a natural language question and identify the relevant entities and properties needed to create a SPARQL query for the university courses knowledge graph.

Guidelines:
1. For each question, extract ALL entities mentioned in the question (courses, categories, research groups, etc.)
2. For each question, extract ALL relevant properties needed to answer the question (has_credits, has_prerequisite_course, has_evaluation_method, etc.)
3. Format your response as a structured JSON object with 'entities' and 'properties' keys
4. Each key should contain an array of strings with the entity or property names
5. Focus ONLY on extracting, not on generating SPARQL queries

Your output should look like:
```json
{
  "entities": ["entity1", "entity2", ...],
  "properties": ["property1", "property2", ...]
}
```
"""

    user_prompt = f"""Question: {row['question']}

Extract all entities and properties from this question that would be needed to generate a SPARQL query for the university courses knowledge graph."""

    # Format entities and properties as JSON arrays
    entities_json = json.dumps(row['entities'])
    properties_json = json.dumps(row['properties'])
    
    asst_prompt = f"""```json
{{
  "entities": {entities_json},
  "properties": {properties_json}
}}
```"""

    if has_chat_template:
        prompt = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
                {"role": "assistant", "content": asst_prompt}
            ],
            tokenize=False,
            add_generation_prompt=False,
            continue_final_message=False
        )
    else:
        prompt = system_prompt + '\n' + user_prompt + '\n' + asst_prompt
    
    return prompt

# Format SPARQL generation data for fine-tuning
def format_sparql_instruction(row):
    system_prompt = """You are a SPARQL generator expert for Indonesian Legal Knowledge Graph (Lex2KG). Your task is to convert the following natural language question to a SPARQL query for the Indonesian legal knowledge graph using the provided entity and property resolutions.

Guidelines:
1. First identify which entities from the list match the question's intent
2. Identify which entities are relevant to the question and select EXACTLY ONE entity ID for each distinct legal concept in the question
3. When multiple entities have similar labels, choose the one that most precisely matches the legal document or concept referenced in the question
4. From the properties list, choose which properties are needed to answer the question
5. Select only the minimum necessary properties required to answer the question correctly
6. Use ALL identified entities and necessary properties in your SPARQL query
7. Use FULL URI NOTATION for entities (e.g., <https://example.org/lex2kg/uu/2013/19>) and PREFIX NOTATION for properties (e.g., lex2kg-o:disahkanDi)
8. Optimize your query by using appropriate SPARQL features (DISTINCT, FILTER, ORDER BY, LIMIT) when needed
9. Return entity URIs directly without using label services
10. Pay attention to legal document hierarchy (UU > Pasal > Ayat) and select the appropriate level entity
11. Return ONLY the raw SPARQL query with no explanations or comments in this format:
   ```sparql
   <your_sparql_query_here>   
   ```

IMPORTANT: Before generating the SPARQL query, provide your step-by-step reasoning about how to construct the query.
"""

    # Format entities and properties matches
    def format_entity_labels(labels):
        result = ""
        for label in labels:
            result += f"- id: {label['id']}, label: {label['label']}\n"
        return result[:-1]  # Remove last extra line
    
    def format_property_labels(labels):
        result = ""
        for label in labels:
            result += f"- id: {label['id']}, label: {label['label']}\n"
        return result[:-1]  # Remove last extra line
    
    entities_labels_formatted = format_entity_labels(row['entities_matches'])
    properties_labels_formatted = format_property_labels(row['properties_matches'])
    
    user_prompt = f"""Question: {row['question']}

Entity Labels:
{entities_labels_formatted}

Property Labels:
{properties_labels_formatted}

SPARQL:"""

    # Format thoughts to be more readable in the assistant's response
    thoughts_formatted = "Let me analyze this step by step:\n\n"
    for thought in row['thoughts']:
        thoughts_formatted += f"{thought}\n"
    
    asst_prompt = f"""{thoughts_formatted}

```sparql
{row['sparql']}
```"""

    if has_chat_template:
        prompt = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
                {"role": "assistant", "content": asst_prompt}
            ],
            tokenize=False,
            add_generation_prompt=False,
            continue_final_message=False
        )
    else:
        prompt = system_prompt + '\n' + user_prompt + '\n' + asst_prompt
    
    return prompt

# Prepare datasets for fine-tuning
entity_prop_train_df['text'] = entity_prop_train_df.apply(format_entity_prop_instruction, axis=1)
entity_prop_val_df['text'] = entity_prop_val_df.apply(format_entity_prop_instruction, axis=1)
entity_prop_train_dataset = Dataset.from_pandas(entity_prop_train_df[['text']])
entity_prop_val_dataset = Dataset.from_pandas(entity_prop_val_df[['text']])

sparql_train_df['text'] = sparql_train_df.apply(format_sparql_instruction, axis=1)
sparql_val_df['text'] = sparql_val_df.apply(format_sparql_instruction, axis=1)
sparql_train_dataset = Dataset.from_pandas(sparql_train_df[['text']])
sparql_val_dataset = Dataset.from_pandas(sparql_val_df[['text']])

print("Prepared datasets for fine-tuning:")
print(f"Entity-property train: {len(entity_prop_train_dataset)} examples")
print(f"Entity-property val: {len(entity_prop_val_dataset)} examples")
print(f"SPARQL train: {len(sparql_train_dataset)} examples")
print(f"SPARQL val: {len(sparql_val_dataset)} examples")

# Sample of formatted data
print("\n============================== Sample of entity-property instruction: ==============================")
print(entity_prop_train_df['text'].iloc[0])

print("\n============================== Sample of SPARQL instruction: ==============================")
print(sparql_train_df['text'].iloc[0])

## Fine-Tune Models

In [ ]:
import torch
import psutil
import gc

def print_gpu_memory():
    """Display detailed GPU memory information"""
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            total_memory = torch.cuda.get_device_properties(i).total_memory / 1024**3  # GB
            allocated_memory = torch.cuda.memory_allocated(i) / 1024**3  # GB
            reserved_memory = torch.cuda.memory_reserved(i) / 1024**3  # GB
            free_memory = total_memory - reserved_memory
            print(f"GPU {i}: Total: {total_memory:.2f} GB | Allocated: {allocated_memory:.2f} GB | Reserved: {reserved_memory:.2f} GB | Free: {free_memory:.2f} GB")
    else:
        print("No GPU available")

def print_system_memory():
    """Display system RAM information"""
    mem = psutil.virtual_memory()
    total_memory = mem.total / 1024**3  # GB
    available_memory = mem.available / 1024**3  # GB
    used_memory = mem.used / 1024**3  # GB
    percent_used = mem.percent
    
    print(f"System Memory: Total: {total_memory:.2f} GB | Used: {used_memory:.2f} GB ({percent_used}%) | Available: {available_memory:.2f} GB")

def print_all_memory():
    """Print both GPU and system memory information"""
    print("\n--- MEMORY STATUS ---")
    print_gpu_memory()
    print_system_memory()
    print("--------------------\n")

# def clear_memory():
#     """Aggressively clear memory"""
#     gc.collect()
#     torch.cuda.empty_cache()
#     print("Memory cleared")
#     print_all_memory()

### Fine-Tune on Template Dataset

In [ ]:
# Fine-tune entity and property extraction model
output_dir_entity_prop = "ft_entity_prop_model_template"

print(f"Fine-tuning entity and property extraction model: {model_name}")
ft_entity_prop_model, ft_entity_prop_tokenizer = finetune_entity_property(
    model_name,
    entity_prop_train_dataset, 
    entity_prop_val_dataset,
    output_dir_entity_prop,
    5,
    "template"
)

!tar -czvf ft_entity_prop_model_template.tar.gz ft_entity_prop_model_template/

# Clear memory
del ft_entity_prop_model, ft_entity_prop_tokenizer
clear_memory()
clear_memory()

# Fine-tune SPARQL generation model
output_dir_sparql = "ft_sparql_model_template"

print(f"Fine-tuning SPARQL generation model: {model_name}")
ft_sparql_model, ft_sparql_tokenizer = finetune_sparql(
    model_name,
    sparql_train_dataset, 
    sparql_val_dataset,
    output_dir_sparql,
    10,
    "template"
)

!tar -czvf ft_sparql_model_template.tar.gz ft_sparql_model_template/

# Clear memory
del ft_sparql_model, ft_sparql_tokenizer
clear_memory()
clear_memory()

### Fine-Tune on random-Walk Dataset

In [ ]:
import time

# Load and process random-walk training data
print("Fine-tuning models on random-walk dataset...")
entity_prop_train_df_rw = load_dataset_entity_prop("train_rw.json")
sparql_train_df_rw = load_dataset_sparql("train_rw.json")

# Split train/validation
entity_prop_train_df_rw, entity_prop_val_df_rw = train_test_split(entity_prop_train_df_rw, test_size=0.2, random_state=42)
sparql_train_df_rw, sparql_val_df_rw = train_test_split(sparql_train_df_rw, test_size=0.2, random_state=42)

print(f"RW Entity-property train/val split: {len(entity_prop_train_df_rw)}/{len(entity_prop_val_df_rw)}")
print(f"RW SPARQL train/val split: {len(sparql_train_df_rw)}/{len(sparql_val_df_rw)}")

# Prepare datasets for fine-tuning
entity_prop_train_df_rw['text'] = entity_prop_train_df_rw.apply(format_entity_prop_instruction, axis=1)
entity_prop_val_df_rw['text'] = entity_prop_val_df_rw.apply(format_entity_prop_instruction, axis=1)
entity_prop_train_dataset_rw = Dataset.from_pandas(entity_prop_train_df_rw[['text']])
entity_prop_val_dataset_rw = Dataset.from_pandas(entity_prop_val_df_rw[['text']])

sparql_train_df_rw['text'] = sparql_train_df_rw.apply(format_sparql_instruction, axis=1)
sparql_val_df_rw['text'] = sparql_val_df_rw.apply(format_sparql_instruction, axis=1)
sparql_train_dataset_rw = Dataset.from_pandas(sparql_train_df_rw[['text']])
sparql_val_dataset_rw = Dataset.from_pandas(sparql_val_df_rw[['text']])

print("Prepared RW datasets for fine-tuning:")
print(f"Entity-property train: {len(entity_prop_train_dataset_rw)} examples")
print(f"Entity-property val: {len(entity_prop_val_dataset_rw)} examples")
print(f"SPARQL train: {len(sparql_train_dataset_rw)} examples")
print(f"SPARQL val: {len(sparql_val_dataset_rw)} examples")

# Fine-tune entity-property extraction model
output_dir_entity_prop_rw = "ft_entity_prop_model_rw"
print(f"Fine-tuning entity and property extraction model on RW data: {model_name}")
ft_entity_prop_model_rw, ft_entity_prop_tokenizer_rw = finetune_entity_property(
    model_name,
    entity_prop_train_dataset_rw, 
    entity_prop_val_dataset_rw,
    output_dir_entity_prop_rw,
    5,
    "rw"
)

!tar -czvf ft_entity_prop_model_rw.tar.gz ft_entity_prop_model_rw/

# Clear memory
del ft_entity_prop_model_rw, ft_entity_prop_tokenizer_rw
clear_memory()
clear_memory()

# Fine-tune SPARQL generation model
output_dir_sparql_rw = "ft_sparql_model_rw"
print(f"Fine-tuning SPARQL generation model on RW data: {model_name}")
ft_sparql_model_rw, ft_sparql_tokenizer_rw = finetune_sparql(
    model_name,
    sparql_train_dataset_rw, 
    sparql_val_dataset_rw,
    output_dir_sparql_rw,
    10,
    "rw"
)

!tar -czvf ft_sparql_model_rw.tar.gz ft_sparql_model_rw/

# Clear memory
del ft_sparql_model_rw, ft_sparql_tokenizer_rw
clear_memory()
clear_memory()

print("RW model fine-tuning complete.")
print(f"RW fine-tuned models saved to {output_dir_entity_prop_rw} and {output_dir_sparql_rw}")

## Initialize Legal Property Retrieval and Verbalization

## Property Retrieval Implementation

In [ ]:
!pip install --upgrade -q protobuf

In [ ]:
import weaviate
import weaviate.classes as wvc

# Legal Property Retrieval class
class LegalPropertyRetrieval:
    def __init__(
        self,
        turtle_file_path: str,
        get_entities_query: str,
        get_properties_query: str,
        embedding_model_name: str = "jinaai/jina-embeddings-v3",
        is_local_client: bool = True,
    ) -> None:
        self.model_embed = SentenceTransformer(embedding_model_name, trust_remote_code=True)
        self.stopwords = set(stopwords.words("english"))
        
        # Load knowledge graph
        self.graph = Graph().parse(turtle_file_path)
        
        # Connect to local Weaviate
        if is_local_client:
            from weaviate.embedded import EmbeddedOptions
            embedded_options = EmbeddedOptions()
            self.client = weaviate.WeaviateClient(embedded_options=embedded_options)
        else:
            self.client = weaviate.connect_to_local(skip_init_checks=True)
        
        self.client.connect()
        
        # Create/get collection
        db_collection_name = "legal_property_db"
        if not self.client.collections.exists(db_collection_name):
            self.collection = self.client.collections.create(
                name=db_collection_name,
                vectorizer_config=wvc.config.Configure.Vectorizer.none(),
            )
            self.is_collection_empty = True
        else:
            self.collection = self.client.collections.get(db_collection_name)
            self.is_collection_empty = False
            
        # Initialize collection with data if empty
        if self.is_collection_empty:
            # Query for entities
            entity_response = self.graph.query(get_entities_query)
            df_entities = pd.DataFrame(entity_response.bindings)
            
            # Query for properties
            property_response = self.graph.query(get_properties_query)
            df_properties = pd.DataFrame(property_response.bindings)
            
            # Process column names
            df_entities.columns = [str(col) for col in df_entities.columns]
            df_properties.columns = [str(col) for col in df_properties.columns]
            
            # Convert all values to strings
            for col in df_entities.columns:
                df_entities[col] = df_entities[col].apply(lambda x: str(x))
            
            for col in df_properties.columns:
                df_properties[col] = df_properties[col].apply(lambda x: str(x))

            # Add labels 
            df_entities["label"] = df_entities["short"].apply(legal_entity_label)
            df_properties["label"] = df_properties["short"].apply(legal_property_label)
            
            # Generate embeddings
            emb_entities = self.model_embed.encode(
                df_entities["label"].tolist(), show_progress_bar=True
            )
            emb_properties = self.model_embed.encode(
                df_properties["label"].tolist(), show_progress_bar=True
            )
            
            # Add data to Weaviate
            legal_df_vectors = {
                "entities": (df_entities, emb_entities),
                "properties": (df_properties, emb_properties),
            }
            
            with self.collection.batch.dynamic() as batch:
                for key, (df, vector) in legal_df_vectors.items():
                    for i, row in df.iterrows():
                        batch.add_object(
                            properties={**row.to_dict(), "type": key},
                            vector=vector[i].tolist(),
                        )

    def _search(self, q: str, type: str = None, k: int = 5) -> pd.DataFrame:
        query_vector = self.model_embed.encode([q])[0]
        response = self.collection.query.hybrid(
            query=q,
            query_properties=["label"],
            vector=query_vector,
            filters=wvc.query.Filter.by_property("type").equal(type) if type else None,
            return_metadata=wvc.query.MetadataQuery(score=True),
            limit=k,
        )
        df = pd.DataFrame(
            [{**o.properties, "score": o.metadata.score} for o in response.objects]
        )
        return df

    def search_entities(self, q: str, k: int = 5) -> pd.DataFrame:
        return self._search(q, type="entities", k=k)

    def search_properties(self, q: str, k: int = 5) -> pd.DataFrame:
        return self._search(q, type="properties", k=k)

    def _preprocess_into_tokens(self, q: str) -> list[str]:
        tokenizer = RegexpTokenizer(r"\w+")
        tokenized = tokenizer.tokenize(q)
        return [tok.lower() for tok in tokenized if tok.lower() not in self.stopwords]

    def _generate_ngrams(self, tokens: list[str]) -> list[str]:
        max_n = min(len(tokens), 3)
        result = []
        for n in range(1, max_n + 1):
            n_grams = ngrams(tokens, n)
            result.extend([" ".join(ng) for ng in n_grams])
        return result

    def get_related_candidates(
        self,
        q: str,
        property_candidates: list[str] = [],
        threshold: float = 0.5,
        k: int = 5,
    ) -> dict[str, list[str]]:
        tokens = self._preprocess_into_tokens(q)
        ngrams = self._generate_ngrams(tokens)
        result = {"entities": [], "properties": []}

        def search(ngram, type, threshold=threshold):
            df_res = self._search(ngram, type=type, k=k)
            if not df_res.empty:
                filtered_results = df_res[df_res["score"] >= threshold][["short", "label"]]
                filtered_results.rename(columns={"short": "id"}, inplace=True)
                return type, filtered_results.to_dict(orient='records')
            return type, []

        for ngram in ngrams + property_candidates:
            for type in result.keys():
                type, df_res = search(ngram, type)
                if df_res:
                    result[type].extend(df_res)
                    result[type] = [dict(t) for t in {tuple(d.items()) for d in result[type]}]

        return result

In [ ]:
# Load the ontology file
turtle_file_path = "data/legal_ontology/final_result.ttl"

# Define queries for entities and properties
get_entities_query = """
SELECT DISTINCT
    (REPLACE(STR(?entity), "https://example.org/lex2kg/", "lex2kg/") AS ?short)
WHERE {
  { 
    ?entity ?predicate ?object. 
    FILTER(isIRI(?entity) && STRSTARTS(STR(?entity), "https://example.org/lex2kg/") && STRSTARTS(STR(?predicate), STR(lex2kg-o:)))
  }
  UNION
  { 
    ?subject ?predicate ?entity. 
    FILTER(isIRI(?entity) && STRSTARTS(STR(?entity), "https://example.org/lex2kg/") && STRSTARTS(STR(?predicate), STR(lex2kg-o:)))
  }
}
"""

get_properties_query = """
SELECT DISTINCT
    (REPLACE(STR(?property), "https://example.org/lex2kg/ontology/", "lex2kg-o:") AS ?short)
WHERE {
  ?subject ?property ?object.
  FILTER(STRSTARTS(STR(?property), STR(lex2kg-o:)))
}
"""
        
# Initialize property retrieval
prop_retrieval = LegalPropertyRetrieval(
    turtle_file_path=turtle_file_path,
    get_entities_query=get_entities_query,
    get_properties_query=get_properties_query,
    embedding_model_name="jinaai/jina-embeddings-v3",
    is_local_client=True,
)

# Initialize verbalization
verbalization = LegalVerbalization(
    model_name="jinaai/jina-embeddings-v3",
    query_model_encode_kwargs={
        "task": "retrieval.query",
        "prompt_name": "retrieval.query",
    },
    passage_model_encode_kwargs={
        "task": "retrieval.passage",
        "prompt_name": "retrieval.passage",
    },
    turtle_file_path=turtle_file_path
)

## Extraction Functions

In [ ]:
# Entity and property extraction function (works for both base and fine-tuned models)
def extract_entities_and_properties(questions, model_path):
    """
    Extract entities and properties from questions using specified model
    """
    # Load model
    model, tokenizer = load_model(model_path)
    pipe = get_pipeline(model, tokenizer)

    system_prompt = """You are an expert entity and property extractor for Indonesian Legal Knowledge Graph (Lex2KG) querying. Your task is to analyze a natural language question and identify the relevant entities and properties needed to create a SPARQL query for the Indonesian legal knowledge graph.

Guidelines:
1. For each question, extract ALL legal entities mentioned in the question (laws, regulations, legal concepts, places, institutions, etc.)
2. For each question, extract ALL relevant properties needed to answer the question (enactment details, document structure, legal relationships, etc.)
3. Format your response as a structured JSON object with 'entities' and 'properties' keys
4. Each key should contain an array of strings with the entity or property names
5. Focus on Indonesian legal terminology and document structure (UU, Pasal, Ayat, Bab, etc.)
6. Consider legal document identifiers (year + number combinations)
7. Include legal process properties (disahkan, tentang, mengingat, menimbang, etc.)
8. Focus ONLY on extracting, not on generating SPARQL queries

Your output should look like:
```json
{
  "entities": ["entity1", "entity2", ...],
  "properties": ["property1", "property2", ...]
}
```
"""

    results = []
    for question in tqdm(questions, desc=f"Extracting entities and properties ({model_path})"):
        user_prompt = f"""Question: {question}

Extract all entities and properties from this question that would be needed to generate a SPARQL query for the Indonesian Legal Knowledge Graph."""

        if has_chat_template:
            prompt = tokenizer.apply_chat_template(
                [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
        else:    
            prompt = system_prompt + '\n' + user_prompt + '\n'
    
        # Generate response
        response = pipe.invoke(prompt)
        
        # Extract JSON from response 
        match = re.search(r"```(?:json)?\s*([\s\S]*?)```", response)
        if match:
            print("gansik")
            json_str = match.group(1).strip()
            try:
                extracted = json.loads(json_str)
                print(extracted)
                results.append({
                    "question": question,
                    "entities": extracted.get("entities", []),
                    "properties": extracted.get("properties", [])
                })
            except json.JSONDecodeError:
                print("json decode error")
                results.append({
                    "question": question,
                    "entities": [],
                    "properties": []
                })
        else:
            print("ga match")
            results.append({
                "question": question,
                "entities": [],
                "properties": []
            })
    
    # Clean up memory
    del model, tokenizer, pipe
    
    return pd.DataFrame(results)

In [ ]:
def get_entity_uris(extraction_df):
    """
    Get the most appropriate entity URIs for extracted entities
    """
    # Load base model
    model, tokenizer = load_model(model_name)
    pipe = get_pipeline(model, tokenizer)

    # Create chat model for langchain
    chat_model = ChatHuggingFace(llm=pipe)

    result_df = extraction_df.copy()
    result_df['entity_uris'] = [[] for _ in range(len(result_df))]

    for idx, row in tqdm(result_df.iterrows(), total=len(result_df), desc="Getting entity URIs"):
        entity_uris = []
        entity_labels = []

        for entity in row['entities']:
            # Search for entities in the property retrieval system
            retrieved_entities = prop_retrieval.search_entities(entity, k=5)
            
            if retrieved_entities.empty:
                continue

            # Define Pydantic model for output parsing
            class Entity(BaseModel):
                """Represents the most appropriate entity ID from the Indonesian Legal Knowledge Graph (Lex2KG)"""
                id: str = Field(..., description="Entity ID")
                
            output_parser = PydanticOutputParser(pydantic_object=Entity)
            format_instructions = output_parser.get_format_instructions()

            chat_prompt_template = ChatPromptTemplate.from_messages(
                [
                    (
                        "system",
                        """Find the most appropriate entity ID for the given entity to answer the question.
Only return a JSON object with an 'id' field. No explanations. E.g., {{"id": "system_administration"}}
{format_instructions}""",
                    ),
                    MessagesPlaceholder("chat_history"),
                    (
                        "human",
                        """Retrieved entities:
{retrieved_entities}
Question: {question}
Entity: {input}
Entity ID:""",
                    ),
                ]
            )

            final_prompt = chat_prompt_template.partial(
                format_instructions=format_instructions,
                retrieved_entities=retrieved_entities[["short", "label"]].to_string(index=False),
                question=row['question'],
            )

            try:
                llm_chain = final_prompt | chat_model | StrOutputParser()
                completion = llm_chain.invoke({"chat_history": [], "input": entity})
                entity = output_parser.parse(completion)
                entity_id = entity.id
                if not entity_id.startswith('ns1:'):
                    entity_id = 'ns1:' + entity_id
                                
                entity_uris.append(entity_id)
            except Exception as e:
                print(f"Error identifying entity URI: {e}")
                print(entity_id)

        result_df.at[idx, 'entity_uris'] = entity_uris

    # Clean up memory
    del model, tokenizer, pipe, chat_model

    return result_df

In [ ]:
def get_entity_prop_labels(extraction_df):
    """
    Get property labels for extracted properties
    """
    result_df = extraction_df.copy()
    result_df['entities_matches'] = [[] for _ in range(len(result_df))]
    result_df['properties_matches'] = [[] for _ in range(len(result_df))]

    for idx, row in tqdm(result_df.iterrows(), total=len(result_df), desc="Getting property labels"):
        property_labels = prop_retrieval.get_related_candidates(
            row['question'], 
            property_candidates=row['entities'] + row['properties'],
            threshold=0.6
        )
        print(property_labels)

        result_df.at[idx, 'entities_matches'] = property_labels['entities']
        result_df.at[idx, 'properties_matches'] = property_labels['properties']

    return result_df

In [ ]:
# Prefix that is needed to execute a SPARQL query
prefix = """prefix lex2kg-o: <https://example.org/lex2kg/ontology/>
prefix xsd: <http://www.w3.org/2001/XMLSchema#>
"""

# Load the knowledge graph
kg = Graph().parse("data/legal_ontology/final_result.ttl")
    
def add_space_before_question_mark(sparql_query):
    """Add a space before the question mark in SPARQL queries."""
    # Add a space before ? that has one space before it
    modified_query = re.sub(r'(?<! )\?', ' ?', sparql_query)
    return modified_query

def extract_query_body(sparql_response):
    # Remove PREFIX declarations
    # This regex finds everything after the last PREFIX declaration
    pattern = r'(?:PREFIX\s+\w*:\s*<[^>]+>\s*)*\s*(SELECT.*)'
    match = re.search(pattern, sparql_query, re.DOTALL | re.IGNORECASE)
    
    if match:
        return match.group(1).strip()
    else:
        return sparql_query.strip()

# SPARQL generation function
def generate_sparql(extraction_df, model_path, try_threshold=5):
    """
    Generate SPARQL queries using specified model
    """
    # Load model
    model, tokenizer = load_model(model_path)
    pipe = get_pipeline(model, tokenizer, max_new_tokens=1024)
    
    result_df = extraction_df.copy()
    result_df['generated_sparql'] = ["" for _ in range(len(result_df))]
    result_df['sparql_results'] = [[] for _ in range(len(result_df))]
    result_df['sparql_generation_attempts'] = [0 for _ in range(len(result_df))]
    result_df['used_verbalization'] = [False for _ in range(len(result_df))]
    
    system_prompt = """You are a SPARQL generator expert for Indonesian Legal Knowledge Graph (Lex2KG). Your task is to convert the following natural language question to a SPARQL query for the Indonesian legal knowledge graph using the provided entity and property resolutions.

Guidelines:
1. First identify which entities from the list match the question's intent
2. Identify which entities are relevant to the question and select EXACTLY ONE entity ID for each distinct legal concept in the question
3. When multiple entities have similar labels, choose the one that most precisely matches the legal document or concept referenced in the question
4. From the properties list, choose which properties are needed to answer the question
5. Select only the minimum necessary properties required to answer the question correctly
6. Use ALL identified entities and necessary properties in your SPARQL query
7. Use FULL URI NOTATION for entities (e.g., <https://example.org/lex2kg/uu/2013/19>) and PREFIX NOTATION for properties (e.g., lex2kg-o:disahkanDi)
8. Optimize your query by using appropriate SPARQL features (DISTINCT, FILTER, ORDER BY, LIMIT) when needed
9. Return entity URIs directly without using label services
10. Pay attention to legal document hierarchy (UU > Pasal > Ayat) and select the appropriate level entity
11. Return ONLY the raw SPARQL query with no explanations or comments in this format:
   ```sparql
   <your_sparql_query_here>
   ```

IMPORTANT: Before generating the SPARQL query, provide your step-by-step reasoning about how to construct the query.
"""

    for idx, row in tqdm(result_df.iterrows(), total=len(result_df), desc=f"Generating SPARQL queries ({model_path})"):
        print(row)
        # Check verbalization score first
        verbalization_score = row.get('verbalization_scores', 0.0)
        
        if verbalization_score >= 0.6:
            # Use verbalization results, skip SPARQL generation
            result_df.at[idx, 'used_verbalization'] = True
            print(f"Question {idx}: Using verbalization (score: {verbalization_score:.3f})")
            continue
        
        # Verbalization score < 0.6, proceed with SPARQL generation
        print(f"Question {idx}: Generating SPARQL (verbalization score: {verbalization_score:.3f})")
        
        # Format entities and properties for the prompt
        def format_entity_labels(labels):
            result = ""
            for label in labels:
                result += f"- id: {label['id']}, label: {label['label']}\n"
            return result[:-1] if result else "No specific entities required"  # Remove last newline
    
        def format_property_labels(labels):
            result = ""
            for label in labels:
                result += f"- id: {label['id']}, label: {label['label']}\n"
            return result[:-1] if result else "No specific properties required"  # Remove last newline
            
        entities_labels_formatted = format_entity_labels(row['entities_matches'])
        properties_labels_formatted = format_property_labels(row['properties_matches'])
        
        user_prompt = f"""Question: {row['question']}

Entities:
{entities_labels_formatted}

Properties:
{properties_labels_formatted}

SPARQL:"""

        # Try generating SPARQL up to try_threshold times
        attempts = 0
        successful = False
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]

        while attempts < try_threshold and not successful:
            attempts += 1
            
            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )

            # Generate response
            response = pipe.invoke(prompt)
            
            # Extract SPARQL from response
            match = re.search(r"```(?:sparql)?\s*([\s\S]*?)```", response)
            if match:
                sparql = match.group(1).strip()
                sparql = extract_query_body(sparql)
                result_df.at[idx, 'generated_sparql'] = sparql
                
                # Add namespace prefixes
                sparql_query = prefix + sparql
                sparql_query = add_space_before_question_mark(sparql_query)
                print(sparql_query)
                
                # Execute query on the knowledge graph
                results = []
                try:
                    for res in kg.query(sparql_query):
                        res_dict = {}
                        for k, v in res.asdict().items():
                            res_dict[str(k)] = str(v)
                        results.append(res_dict)
                    
                    if results:
                        result_df.at[idx, 'sparql_results'] = results
                        successful = True
                        print(f"  Attempt {attempts}: Success")
                    else:
                        print(f"  Attempt {attempts}: Failed - Empty results")
                        messages.append(
                            {"role": "assistant", "content": response}
                        )
                        messages.append(
                            {"role": "user", "content": f"""The SPARQL query you generated produced empty results. 
Please generate a better query using different properties, entities, or query structure to address the original question."""}
                        )
                except Exception as e:
                    print(f"  Attempt {attempts}: Error - {e}")
                    messages.append(
                        {"role": "assistant", "content": response}
                    )
                    messages.append(
                        {"role": "user", "content": f"""The SPARQL query you generated returned an error. 
Please generate a better query using different properties, entities, or query structure to address the original question."""}
                    )
            else:
                print(f"  Attempt {attempts}: No SPARQL found in response")
                messages.append(
                    {"role": "assistant", "content": response}
                )
                messages.append(
                    {"role": "user", "content": """I need a SPARQL query to answer my question, but no valid SPARQL code was found in your response.
Please provide a complete, executable SPARQL query enclosed in triple backticks (```).```"""}
                )
        
        result_df.at[idx, 'sparql_generation_attempts'] = attempts
    
    # Clean up memory
    del model, tokenizer, pipe
    
    return result_df

In [ ]:
def verbalize_entities(extraction_df):
    """
    Run verbalization for the identified entities
    """
    result_df = extraction_df.copy()
    result_df['verbalization_results'] = [[] for _ in range(len(result_df))]
    result_df['verbalization_scores'] = [0.0 for _ in range(len(result_df))]
    
    for idx, row in tqdm(result_df.iterrows(), total=len(result_df), desc="Verbalizing entities"):
        best_result = []
        best_score = 0.0
        
        for entity_uri in row['entity_uris']:
            try:
                result, score = verbalization.run(row['question'], entity_uri, True)
                if score > best_score:
                    best_result = result
                    best_score = score
            except Exception as e:
                print(f"Error in verbalization: {e}")
        
        result_df.at[idx, 'verbalization_results'] = best_result
        result_df.at[idx, 'verbalization_scores'] = best_score
    
    display(result_df)
    return result_df

## Main Evaluation Pipeline

In [ ]:
def compare_two_dataframes(df1: pd.DataFrame, df2: pd.DataFrame) -> Dict[str, float]:
    # df1: DataFrame for ground truth
    # df2: DataFrame for predicted
    if len(df1.columns) != len(df2.columns):
        return {
                    'jaccard': 0,
                    'recall': 0,
                    'precision': 0,
                    'f1': 0,
                    'tp': 0,
                    'fp': 0,
                    'fn': 0,
                    'tn': 0
                }

    set1, set2 = set(), set()
    for _, row in df1.iterrows():
        row = list(row)
        row = sorted(row)
        row = tuple(row)
        set1.add(row)

    for _, row in df2.iterrows():
        row = list(row)
        row = sorted(row)
        row = tuple(row)
        set2.add(row)
    
    jaccard = len(set1 & set2) / len(set1 | set2) if len(set1 | set2) > 0 else 0
    # recall = correct retrieved / all ground truth
    recall = len(set1 & set2) / len(set1) if len(set1) > 0 else 0
    # precision = correct retrieved / retrieved answers
    precision = len(set1 & set2) / len(set2) if len(set2) > 0 else 0
    # f1 score = 2 x prec x recall / (prec + recall)
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    # TP, TN, FP, FN computation (might be useful for computing micro metrics)
    tp = len(set1 & set2)
    fp = len(set2) - tp
    fn = len(set1) - tp
    total_pairs = len(set1) + len(set2) - tp
    tn = total_pairs - (tp + fp + fn)

    return {
        'jaccard': jaccard,
        'recall': recall,
        'precision': precision,
        'f1': f1,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'tn': tn
    }

In [ ]:
def evaluate_batch_results(results_df, ground_truth_queries, questions, complexities):
    """
    Evaluate the batch results from the FROG pipeline with complexity-based grouping
    """
    # Load the knowledge graph
    evaluation_results = []
    
    print(f"Evaluating {len(results_df)} questions...")
    
    for idx, row in results_df.iterrows():
        question = row['question']
        generated_query = row.get('generated_sparql', '')
        sparql_results = row.get('sparql_results', [])
        verbalization_results = row.get("verbalization_results", [])
        verbalization_score = row.get('verbalization_scores', 0.0)
        used_verbalization = row.get('used_verbalization', False)
        ground_truth_query = ground_truth_queries[idx]
        complexity = complexities[idx]  # Get the complexity for this question
        
        print(f"\n=== Question {idx + 1}: {question} (Complexity: {complexity}) ===")
        
        result_entry = {
            "question_id": idx,
            "question_text": question,
            "complexity": complexity,  # Add complexity to results
            "ground_truth_query": ground_truth_query,
            "generated_query": generated_query,
            "entities": row.get('entities', []),
            "properties": row.get('properties', []),
            "entity_uris": row.get('entity_uris', []),
            "verbalization_score": verbalization_score,
            "used_verbalization": used_verbalization,
            "sparql_generation_attempts": row.get('sparql_generation_attempts', 0),
            "approach_used": "verbalization" if used_verbalization else "sparql_generation",
            "metrics": {},
            "error": None
        }
        
        try:
            # Execute ground truth query
            try:
                # Add namespace prefixes
                sparql_query = prefix + ground_truth_query
                sparql_query = add_space_before_question_mark(sparql_query)
                
                # Get results
                ground_truth_results = []
                for res in kg.query(sparql_query):
                    res_dict = {}
                    for k, v in res.asdict().items():
                        res_dict[str(k)] = str(v)
                    ground_truth_results.append(res_dict)
                
                ground_truth_df = pd.DataFrame(ground_truth_results)
            except Exception as e:
                print(f"Error executing ground truth query: {e}")
                ground_truth_df = pd.DataFrame()
            
            print(f"Ground truth shape: {ground_truth_df.shape}")
            if not ground_truth_df.empty:
                print(ground_truth_df.head())

            # Determine which results to use
            if used_verbalization and verbalization_results:
                # Use verbalization results
                pred_df = pd.DataFrame(verbalization_results)
                print(f"Using verbalization results (score: {verbalization_score:.3f})")
            elif sparql_results:
                # Use SPARQL results
                pred_df = pd.DataFrame(sparql_results)
                print(f"Using SPARQL results (attempts: {row.get('sparql_generation_attempts', 0)})")
            else:
                pred_df = pd.DataFrame()
                print("No results available")
            
            if not pred_df.empty and not ground_truth_df.empty:
                print(f"Prediction shape: {pred_df.shape}")
                print(pred_df.head())
                
                # Compare results
                metrics = compare_two_dataframes(ground_truth_df, pred_df)
                result_entry["metrics"] = metrics
                
                print(f"Metrics: Jaccard={metrics['jaccard']:.3f}, Precision={metrics['precision']:.3f}, Recall={metrics['recall']:.3f}")
            else:
                print("No results to evaluate")
                result_entry["metrics"] = {
                    'jaccard': 0, 'recall': 0, 'precision': 0, 'f1': 0,
                    'tp': 0, 'fp': 0, 'fn': 0, 'tn': 0
                }
                result_entry["error"] = "No results generated"
                
        except Exception as e:
            print(f"Error evaluating question {idx}: {e}")
            result_entry["error"] = str(e)
            result_entry["metrics"] = {
                'jaccard': 0, 'recall': 0, 'precision': 0, 'f1': 0,
                'tp': 0, 'fp': 0, 'fn': 0, 'tn': 0
            }
        
        evaluation_results.append(result_entry)
    
    return evaluation_results

In [ ]:
def calculate_summary_statistics(evaluation_results):
    """Calculate summary statistics from evaluation results, grouped by complexity"""    
    # Calculate average metrics - Include all results with metrics, even if they have errors
    valid_results = [r for r in evaluation_results if r["metrics"]]
    
    if not valid_results:
        print("No valid results to summarize")
        return {}
    
    # Calculate overall metrics
    avg_metrics = {}
    for metric in valid_results[0]["metrics"].keys():
        avg_metrics[metric] = sum(r["metrics"][metric] for r in valid_results) / len(valid_results)
    
    # Calculate success rates
    total_questions = len(evaluation_results)
    successful_sparql = len([r for r in evaluation_results if r["generated_query"] and not r["error"]])
    successful_results = len([r for r in evaluation_results if r["metrics"]["f1"] > 0])
    
    # Verbalization vs SPARQL statistics
    verbalization_used = len([r for r in evaluation_results if r.get("used_verbalization", False)])
    sparql_attempts = [r.get("sparql_generation_attempts", 0) for r in evaluation_results if not r.get("used_verbalization", False)]
    avg_sparql_attempts = sum(sparql_attempts) / len(sparql_attempts) if sparql_attempts else 0
    
    # Entity and property extraction stats
    avg_entities = sum(len(r["entities"]) for r in evaluation_results) / total_questions
    avg_properties = sum(len(r["properties"]) for r in evaluation_results) / total_questions
    avg_entity_uris = sum(len(r["entity_uris"]) for r in evaluation_results) / total_questions
    avg_verbalization_score = sum(r["verbalization_score"] for r in evaluation_results) / total_questions
    
    # Group results by complexity
    complexity_groups = {}
    for r in evaluation_results:
        complexity = r.get("complexity", "unknown")
        if complexity not in complexity_groups:
            complexity_groups[complexity] = []
        complexity_groups[complexity].append(r)
    
    # Calculate metrics by complexity
    complexity_metrics = {}
    for complexity, results in complexity_groups.items():
        if not results:
            continue
            
        valid_results_in_group = [r for r in results if r["metrics"]]
        if not valid_results_in_group:
            continue
            
        # Calculate averages for this complexity group
        group_metrics = {}
        for metric in valid_results_in_group[0]["metrics"].keys():
            group_metrics[metric] = sum(r["metrics"][metric] for r in valid_results_in_group) / len(valid_results_in_group)
        
        # Calculate success rates for this group
        group_total = len(results)
        group_successful_sparql = len([r for r in results if r["generated_query"] and not r["error"]])
        group_successful_results = len([r for r in results if r["metrics"]["f1"] > 0])
        
        complexity_metrics[complexity] = {
            "total_questions": group_total,
            "successful_sparql": group_successful_sparql,
            "successful_results": group_successful_results,
            "sparql_success_rate": group_successful_sparql / group_total if group_total > 0 else 0,
            "results_success_rate": group_successful_results / group_total if group_total > 0 else 0,
            "average_metrics": group_metrics
        }
    
    summary = {
        "total_questions": total_questions,
        "successful_sparql_generation": successful_sparql,
        "successful_results": successful_results,
        "sparql_success_rate": successful_sparql / total_questions,
        "results_success_rate": successful_results / total_questions,
        "verbalization_used_count": verbalization_used,
        "verbalization_usage_rate": verbalization_used / total_questions,
        "avg_sparql_attempts": avg_sparql_attempts,
        "average_metrics": avg_metrics,
        "avg_entities_per_question": avg_entities,
        "avg_properties_per_question": avg_properties,
        "avg_entity_uris_per_question": avg_entity_uris,
        "avg_verbalization_score": avg_verbalization_score,
        "complexity_based_metrics": complexity_metrics  # Add complexity-based metrics
    }
    
    return summary

In [ ]:
def display_model_results(title, summary_stats):
    print(f"\n{title}")
    print("=" * len(title))
    
    print(f"Total Questions Processed: {summary_stats['total_questions']}")
    print(f"SPARQL Generation Success Rate: {summary_stats['sparql_success_rate']:.1%}")
    print(f"Results Success Rate: {summary_stats['results_success_rate']:.1%}")
    
    print("\nPipeline Component Averages:")
    print(f"  Entities per question: {summary_stats['avg_entities_per_question']:.2f}")
    print(f"  Properties per question: {summary_stats['avg_properties_per_question']:.2f}")
    print(f"  Entity URIs per question: {summary_stats['avg_entity_uris_per_question']:.2f}")
    print(f"  Verbalization score: {summary_stats['avg_verbalization_score']:.4f}")
    print(f"  Verbalization usage rate: {summary_stats['verbalization_usage_rate']:.1%}")
    print(f"  Average SPARQL attempts: {summary_stats['avg_sparql_attempts']:.2f}")
    
    if summary_stats.get("average_metrics"):
        print("\nAverage Evaluation Metrics:")
        for metric, score in summary_stats["average_metrics"].items():
            print(f"  {metric.upper()}: {score:.4f}")
    
    # Display complexity-based metrics
    if summary_stats.get("complexity_based_metrics"):
        print("\nResults by Question Complexity:")
        for complexity, metrics in summary_stats["complexity_based_metrics"].items():
            print(f"\n  {complexity.upper()} Questions ({metrics['total_questions']} questions):")
            print(f"    Success Rate: {metrics['results_success_rate']:.1%}")
            print(f"    Jaccard: {metrics['average_metrics']['jaccard']:.4f}")
            print(f"    F1 Score: {metrics['average_metrics']['f1']:.4f}")
            print(f"    Precision: {metrics['average_metrics']['precision']:.4f}")
            print(f"    Recall: {metrics['average_metrics']['recall']:.4f}")

## Perform Evaluation


In [ ]:
def display_comparative_results(all_results):
    """Display comparative results for all workflows on both datasets"""
    print("\n" + "=" * 80)
    print("COMPREHENSIVE COMPARATIVE ANALYSIS")
    print("=" * 80)
    
    workflows = ["base", "finetuned_template", "finetuned_rw"]
    workflow_names = {
        "base": "Base Model",
        "finetuned_template": "Fine-tuned (Template)",
        "finetuned_rw": "Fine-tuned (Random-Walk)"
    }
    
    datasets = ["template", "rw"]
    dataset_names = {
        "template": "Template Dataset",
        "rw": "Random-Walk Dataset"
    }
    
    # Compare overall F1 performance across datasets and workflows
    print("\nOVERALL PERFORMANCE (F1 SCORE)")
    print("-" * 40)
    
    # Create a table header
    header = "Workflow"
    for dataset in datasets:
        header += f" | {dataset_names[dataset]}"
    print(header)
    print("-" * len(header))
    
    # Fill the table with F1 scores
    for workflow in workflows:
        row = workflow_names[workflow]
        for dataset in datasets:
            if dataset in all_results and workflow in all_results[dataset]:
                f1_score = all_results[dataset][workflow]["summary_stats"]["average_metrics"]["f1"]
                row += f" | {f1_score:.4f}"
            else:
                row += " | N/A"
        print(row)
    
    # Compare overall Jaccard performance
    print("\nOVERALL PERFORMANCE (JACCARD SIMILARITY)")
    print("-" * 40)
    
    # Create a table header
    header = "Workflow"
    for dataset in datasets:
        header += f" | {dataset_names[dataset]}"
    print(header)
    print("-" * len(header))
    
    # Fill the table with Jaccard scores
    for workflow in workflows:
        row = workflow_names[workflow]
        for dataset in datasets:
            if dataset in all_results and workflow in all_results[dataset]:
                jaccard_score = all_results[dataset][workflow]["summary_stats"]["average_metrics"]["jaccard"]
                row += f" | {jaccard_score:.4f}"
            else:
                row += " | N/A"
        print(row)
    
    # Compare performance by complexity
    print("\nPERFORMANCE BY COMPLEXITY (F1 SCORE)")
    print("-" * 40)
    
    complexities = ["basic", "intermediate", "advanced"]
    
    for complexity in complexities:
        print(f"\n{complexity.upper()} QUESTIONS:")
        
        # Create a table header for this complexity
        header = "Workflow"
        for dataset in datasets:
            header += f" | {dataset_names[dataset]}"
        print(header)
        print("-" * len(header))
        
        # Fill the table with F1 scores for this complexity
        for workflow in workflows:
            row = workflow_names[workflow]
            for dataset in datasets:
                if (dataset in all_results and 
                    workflow in all_results[dataset] and 
                    "complexity_based_metrics" in all_results[dataset][workflow]["summary_stats"] and
                    complexity in all_results[dataset][workflow]["summary_stats"]["complexity_based_metrics"]):
                    
                    f1_score = all_results[dataset][workflow]["summary_stats"]["complexity_based_metrics"][complexity]["average_metrics"]["f1"]
                    row += f" | {f1_score:.4f}"
                else:
                    row += " | N/A"
            print(row)
    
    # Compare performance by complexity (Jaccard)
    print("\nPERFORMANCE BY COMPLEXITY (JACCARD SIMILARITY)")
    print("-" * 40)
    
    for complexity in complexities:
        print(f"\n{complexity.upper()} QUESTIONS:")
        
        # Create a table header for this complexity
        header = "Workflow"
        for dataset in datasets:
            header += f" | {dataset_names[dataset]}"
        print(header)
        print("-" * len(header))
        
        # Fill the table with Jaccard scores for this complexity
        for workflow in workflows:
            row = workflow_names[workflow]
            for dataset in datasets:
                if (dataset in all_results and 
                    workflow in all_results[dataset] and 
                    "complexity_based_metrics" in all_results[dataset][workflow]["summary_stats"] and
                    complexity in all_results[dataset][workflow]["summary_stats"]["complexity_based_metrics"]):
                    
                    jaccard_score = all_results[dataset][workflow]["summary_stats"]["complexity_based_metrics"][complexity]["average_metrics"]["jaccard"]
                    row += f" | {jaccard_score:.4f}"
                else:
                    row += " | N/A"
            print(row)
    
    # Compare success rates
    print("\nSUCCESS RATES")
    print("-" * 40)
    
    # Create a table header
    header = "Workflow"
    for dataset in datasets:
        header += f" | {dataset_names[dataset]}"
    print(header)
    print("-" * len(header))
    
    # Fill the table with success rates
    for workflow in workflows:
        row = workflow_names[workflow]
        for dataset in datasets:
            if dataset in all_results and workflow in all_results[dataset]:
                success_rate = all_results[dataset][workflow]["summary_stats"]["results_success_rate"]
                row += f" | {success_rate:.1%}"
            else:
                row += " | N/A"
        print(row)
def save_comprehensive_results(all_results):
    """Save comprehensive evaluation results to files"""
    print("\nSaving comprehensive results...")
    
    # Prepare results for saving
    save_results = {
        "evaluation_metadata": {
            "model_info": {
                "base_model_name": model_name,
                "finetuned_template_entity_model_path": "ft_entity_prop_model_template",
                "finetuned_template_sparql_model_path": "ft_sparql_model_template",
                "finetuned_rw_entity_model_path": "ft_entity_prop_model_rw",
                "finetuned_rw_sparql_model_path": "ft_sparql_model_rw"
            },
            "evaluation_date": "2025-05-26"
        }
    }
    
    workflows = ["base", "finetuned_template", "finetuned_rw"]
    datasets = ["template", "rw"]
    
    for dataset in datasets:
        save_results[dataset] = {}
        for workflow in workflows:
            if dataset in all_results and workflow in all_results[dataset]:
                # Remove DataFrame objects as they're not JSON serializable
                results_copy = deepcopy(all_results[dataset][workflow])
                if "results_df" in results_copy:
                    results_copy["pipeline_results"] = results_copy["results_df"].to_dict('records')
                    del results_copy["results_df"]
                
                save_results[dataset][workflow] = results_copy
    
    # Save complete results
    with open('frog_comprehensive_evaluation_results.json', 'w') as f:
        json.dump(save_results, f, indent=2)
    
    print("Comprehensive results saved to 'frog_comprehensive_evaluation_results.json'")

In [ ]:
# Main execution code for comprehensive evaluation
print("Starting comprehensive FROG evaluation...")
print("=" * 60)

# Run comprehensive evaluation
print("\nRunning comprehensive evaluation")
# Define model paths for all workflows
workflows = {
    "base": {
        "entity_model_path": model_name,
        "sparql_model_path": model_name,
        "name": "Base Model"
    },
    "finetuned_template": {
        "entity_model_path": "ft_entity_prop_model_template",
        "sparql_model_path": "ft_sparql_model_template",
        "name": "Fine-tuned (Template)"
    },
    "finetuned_rw": {
        "entity_model_path": "ft_entity_prop_model_rw",
        "sparql_model_path": "ft_sparql_model_rw",
        "name": "Fine-tuned (Random-Walk)"
    }
}

# Define test datasets
datasets = {
    "template": {
        "path": "test_template.json",
        "name": "Template Dataset"
    },
    "rw": {
        "path": "test_rw.json",
        "name": "Random-Walk Dataset"
    }
}

# Store all results
all_results = {}

# Evaluate each workflow on each dataset
for dataset_key, dataset_info in datasets.items():
    print(f"\n{'=' * 80}")
    print(f"EVALUATING ON {dataset_info['name'].upper()}")
    print(f"{'=' * 80}")
    
    # Load test data
    test_df = load_test_data(dataset_info['path'])
    questions = test_df['question'].tolist()[:5]
    ground_truth_queries = test_df['sparql'].tolist()[:5]
    complexities = test_df['complexity'].tolist()[:5]
    
    print(f"Loaded {len(questions)} test questions from {dataset_info['name']}")
    
    # Initialize results for this dataset
    all_results[dataset_key] = {}
    
    # Run each workflow
    for workflow_key, workflow_info in workflows.items():
        print(f"\n{'-' * 60}")
        print(f"EVALUATING {workflow_info['name'].upper()} ON {dataset_info['name'].upper()}")
        print(f"{'-' * 60}")
        
        # Run the pipeline
        entity_model_path=workflow_info['entity_model_path']
        sparql_model_path=workflow_info['sparql_model_path']

        # Step 1: Extract entities and properties
        print(f"Step 1: Extracting entities and properties using {entity_model_path}")
        entity_property_extraction = extract_entities_and_properties(questions, entity_model_path)
        clear_memory()
        clear_memory()
        
        # Step 2: Get entity URIs
        print(f"Step 2: Identifying entity URIs")
        entity_uri_df = get_entity_uris(entity_property_extraction)
        clear_memory()
        clear_memory()
        
        # Step 3: Get property labels
        print(f"Step 3: Getting property labels")
        property_labels_df = get_entity_prop_labels(entity_uri_df)
        
        # Step 4: Run verbalization
        print(f"Step 4: Running verbalization")
        verbalization_df = verbalize_entities(property_labels_df)
        
        # Step 5: Generate SPARQL (with verbalization threshold check)
        print(f"Step 5: Generating SPARQL queries using {sparql_model_path}")
        results_df = generate_sparql(verbalization_df, sparql_model_path, try_threshold=5)
        clear_memory()
        clear_memory()
        
        # Evaluate results
        evaluation_results = evaluate_batch_results(
            results_df, 
            ground_truth_queries,
            questions,
            complexities
        )
        
        # Calculate summary statistics
        summary_stats = calculate_summary_statistics(evaluation_results)
        
        # Store results
        all_results[dataset_key][workflow_key] = {
            "results_df": results_df,
            "evaluation_results": evaluation_results,
            "summary_stats": summary_stats
        }
        
        # Display results
        display_model_results(f"{workflow_info['name']} on {dataset_info['name']}", summary_stats)
        
# Display comparative results
print("\nDisplaying comparative results")
display_comparative_results(all_results)

# Save results
print("\nSaving results")
save_comprehensive_results(all_results)

print("\nComprehensive evaluation complete!")

In [ ]:
for q in ground_truth_queries:
    print(q)